# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.004922,0.001773,0.993304,0.004134,0.002593,0.993273
1,0.997228,0.000463,0.002309,0.995591,0.000405,0.004004
2,0.002369,0.000604,0.997026,0.001958,0.000503,0.997538
3,0.004656,0.003362,0.991982,0.002466,0.002391,0.995142
4,0.995353,0.000016,0.004632,0.994178,0.000003,0.005819


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012452,0.003199,0.984349,0.011048,0.004491,0.984461
1,0.472116,0.002331,0.525553,0.499186,0.001896,0.498918
2,0.998690,0.001012,0.000298,0.997826,0.001594,0.000579
3,0.980854,0.000018,0.019128,0.987434,0.000009,0.012557
4,0.006765,0.001847,0.991388,0.006876,0.001972,0.991152


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
cols_use = ['lgbm_0', 'lgbm_1', 'lgbm_2']

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.0, 10.0, log=False)
    w1 = trial.suggest_float('weight_class_1', 0.0, 10.0, log=False)
    w2 = trial.suggest_float('weight_class_2', 0.0, 10.0, log=False)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, cols_use].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=1000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-05 15:03:12,753] A new study created in memory with name: no-name-10d19b54-52d8-41f2-96d4-0fecd001b340
Best trial: 3. Best value: 0.944814:   1%|█▍                                                                                                                                     | 11/1000 [00:00<00:31, 31.20it/s]

[I 2026-07-05 15:03:12,923] Trial 9 finished with value: 0.9378633900057229 and parameters: {'weight_class_0': 0.30702977718320756, 'weight_class_1': 0.6933869563074491, 'weight_class_2': 5.125333506862581}. Best is trial 9 with value: 0.9378633900057229.
[I 2026-07-05 15:03:12,928] Trial 5 finished with value: 0.8433900855586592 and parameters: {'weight_class_0': 2.5233907004976674, 'weight_class_1': 0.3736195578831414, 'weight_class_2': 2.3286421882156247}. Best is trial 9 with value: 0.9378633900057229.
[I 2026-07-05 15:03:12,941] Trial 7 finished with value: 0.8355775924344732 and parameters: {'weight_class_0': 9.094137661097395, 'weight_class_1': 3.4816009989986494, 'weight_class_2': 0.8060038630659616}. Best is trial 9 with value: 0.9378633900057229.
[I 2026-07-05 15:03:12,942] Trial 0 finished with value: 0.8977311578968941 and parameters: {'weight_class_0': 5.822720498237296, 'weight_class_1': 7.165158413308951, 'weight_class_2': 8.24689409830422}. Best is trial 9 with value: 0

[I 2026-07-05 15:03:13,093] Trial 12 finished with value: 0.9225897479471769 and parameters: {'weight_class_0': 4.360737476174032, 'weight_class_1': 9.158154524477787, 'weight_class_2': 9.34557340983936}. Best is trial 3 with value: 0.9448142829901585.
[I 2026-07-05 15:03:13,098] Trial 13 finished with value: 0.9060997873112244 and parameters: {'weight_class_0': 4.012823652685961, 'weight_class_1': 4.0778213094312115, 'weight_class_2': 9.250176777669491}. Best is trial 3 with value: 0.9448142829901585.
[I 2026-07-05 15:03:13,124] Trial 14 finished with value: 0.9134439977284027 and parameters: {'weight_class_0': 2.8692085640490665, 'weight_class_1': 3.372292518041746, 'weight_class_2': 7.728976207825054}. Best is trial 3 with value: 0.9448142829901585.
[I 2026-07-05 15:03:13,135] Trial 16 finished with value: 0.9384293053874986 and parameters: {'weight_class_0': 2.967894432381838, 'weight_class_1': 9.965713454255347, 'weight_class_2': 9.783855764280446}. Best is trial 3 with value: 0.9

Best trial: 37. Best value: 0.948704:   4%|████▋                                                                                                                                 | 35/1000 [00:00<00:19, 49.17it/s]

[I 2026-07-05 15:03:13,298] Trial 24 finished with value: 0.9426726388726272 and parameters: {'weight_class_0': 1.505891414167728, 'weight_class_1': 5.43031946724064, 'weight_class_2': 7.189308900020879}. Best is trial 3 with value: 0.9448142829901585.
[I 2026-07-05 15:03:13,328] Trial 25 finished with value: 0.9450440155441119 and parameters: {'weight_class_0': 1.3576811884301294, 'weight_class_1': 6.144563213372406, 'weight_class_2': 7.085021898362447}. Best is trial 25 with value: 0.9450440155441119.
[I 2026-07-05 15:03:13,346] Trial 26 finished with value: 0.9407781916751796 and parameters: {'weight_class_0': 1.672130707644831, 'weight_class_1': 5.4669510543666435, 'weight_class_2': 7.1793837860603755}. Best is trial 25 with value: 0.9450440155441119.
[I 2026-07-05 15:03:13,350] Trial 27 finished with value: 0.9418513006491461 and parameters: {'weight_class_0': 1.499438073169074, 'weight_class_1': 5.152263994162983, 'weight_class_2': 6.826993338365206}. Best is trial 25 with value:

...

[I 2026-07-05 15:03:32,097] Trial 969 finished with value: 0.947049467062156 and parameters: {'weight_class_0': 1.2420051254298112, 'weight_class_1': 8.011166200200792, 'weight_class_2': 6.483423602060442}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,108] Trial 970 finished with value: 0.9482427128978905 and parameters: {'weight_class_0': 1.0027708835492353, 'weight_class_1': 8.54569073507172, 'weight_class_2': 6.2280164907133395}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,127] Trial 971 finished with value: 0.9458268304894428 and parameters: {'weight_class_0': 1.3866032027700284, 'weight_class_1': 8.57908684824255, 'weight_class_2': 6.216449307877622}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,170] Trial 972 finished with value: 0.9475154359461184 and parameters: {'weight_class_0': 1.1951505987739643, 'weight_class_1': 8.602864002393183, 'weight_class_2': 6.503417017543227}. Best is trial 673 with 

Best trial: 673. Best value: 0.949875:  99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎ | 987/1000 [00:19<00:00, 43.36it/s]

[I 2026-07-05 15:03:32,336] Trial 979 finished with value: 0.949425603148955 and parameters: {'weight_class_0': 0.6617036663743333, 'weight_class_1': 8.603004382753085, 'weight_class_2': 9.98906018272401}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,345] Trial 978 finished with value: 0.9497266798873666 and parameters: {'weight_class_0': 0.6370937182222245, 'weight_class_1': 8.617749228354093, 'weight_class_2': 6.2377517147305355}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,356] Trial 980 finished with value: 0.949743027168518 and parameters: {'weight_class_0': 0.6341762871593332, 'weight_class_1': 9.45933723949305, 'weight_class_2': 6.227736260673934}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,372] Trial 981 finished with value: 0.9497646765354109 and parameters: {'weight_class_0': 0.6456395881118843, 'weight_class_1': 9.427383542406025, 'weight_class_2': 6.518368122711991}. Best is trial 673 with v

Best trial: 673. Best value: 0.949875: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:19<00:00, 50.06it/s]

[I 2026-07-05 15:03:32,548] Trial 989 finished with value: 0.9497128677471349 and parameters: {'weight_class_0': 0.6431022319293364, 'weight_class_1': 9.445070910992431, 'weight_class_2': 6.185250119629408}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,563] Trial 988 finished with value: 0.9293382772274481 and parameters: {'weight_class_0': 2.84335527090625, 'weight_class_1': 8.85418211709563, 'weight_class_2': 6.153023147096167}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,588] Trial 990 finished with value: 0.9494441884861572 and parameters: {'weight_class_0': 0.3971535504388346, 'weight_class_1': 8.861804956873161, 'weight_class_2': 6.169604938445288}. Best is trial 673 with value: 0.949874946749033.
[I 2026-07-05 15:03:32,603] Trial 991 finished with value: 0.9495032646696197 and parameters: {'weight_class_0': 0.4061202987778293, 'weight_class_1': 8.856645953624755, 'weight_class_2': 6.118057253311774}. Best is trial 673 with v

In [29]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, cols_use].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [30]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [31]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.1.0_lightgbm_submission.csv', index=False)

In [32]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
